# Feature Extraction Pipeline
This pipeline holds the preprocessing of the `Standard calibration in culture media_extended.xlsx` and the feature extraction process. The results will be 3 `.csv` files, each holding the features for 3 types experiments, specific for this dataset:
1. the core features suite
2. the extended core features suite
3. the experimental features suite

### Dataset structure
- There are 42 signals, stored as column vectors
- The first signal, from the first column, represents the potential - denoted with $E$ and measured in $V$ (volts) - and this potential was applied in all experiments, ensuring the same potential window.
- The other signals represent the cureents - denoted with $I$ and measured in $\mu A$ (microampers) - characteristic to the initial potential $E$ and to the concentration values - denoted with $c$ and measured in $\mu M$ (micromolars), for which the experiment was performed.
- For each concentration were performed a number of readings, as shown bellow:

| $c$ [$\mu M$]| Number of readings |
| -------- | ------- |
| 0 | 1 (the blank signal)|
| 100 | 3 |
| 50 | 3 |
| 25 | 3 |
| 15 | 3 |
| 10 | 3 |
| 7.5 | 3 |
| 5 | 3 |
| 2.5 | 2 |
| 1 | 3 |
| 0.5 | 6 |
| 0.5 | 6 |
| 0.25 | 3 |
| 0.1 | 5 |

### The Problem we are trying to solve
- Create a ML model that predicts the concentration value (in $\mu M$) of a given voltammogram signal.
- The current signal, represented in the voltammogram, can be seen as a function of potential. Therefore, the entire problem can be written as:
$$c = I(E)$$

In [1]:
# imports

import pandas as pd
import csv
import os
from voltammogram_signal import Signal

In [2]:
# reading the Standard calibration media dataset
df = pd.read_excel('datasets/Standard calibration in culture media_extended.xlsx', sheet_name='Raw data')
df.head()

,E0_Blank_culture media MH - moving average baseline,Unnamed: 1,Pyo 100 uM_MH - moving average baseline,Unnamed: 3,Unnamed: 4,Pyo 50 uM_MH - moving average baseline,Unnamed: 6,Unnamed: 7,Pyo 25 uM_MH - moving average baseline,Unnamed: 9,...,Unnamed: 32,Unnamed: 33,Pyo 0.25 uM_MH - moving average baseline,Unnamed: 35,Unnamed: 36,Pyo 0.1 uM_MH - moving average baseline,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41
0,V,µA,µA,µA,µA,µA,µA,µA,µA,µA,...,µA,µA,µA,µA,µA,µA,µA,µA,µA,µA
1,-0.600097,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-0.595262,0.063593,0.098656,0.109374,0.115718,0.082966,0.110628,0.104234,0.088812,0.082304,...,0.079479,0.075286,0.075833,0.076125,0.079187,0.083927,0.078385,0.110979,0.051843,0.066755
3,-0.590427,0.054031,0.121318,0.130725,0.148006,0.099352,0.122679,0.113728,0.090737,0.073225,...,0.04127,0.070401,0.071166,0.054775,0.054862,0.065697,0.060739,0.079771,0.044581,0.064057
4,-0.585592,0.033967,0.136981,0.152074,0.173293,0.115738,0.113729,0.119721,0.085662,0.085147,...,0.038062,0.048015,0.045499,0.040424,0.044536,0.043967,0.064092,0.062562,0.030319,0.047358


Constructing the Signal objects

In [3]:
Signal.set_common_potential_E(df.iloc[1:, 0].values)
Signal.set_common_baseline_I(df.iloc[1:, 1].values)

signals = []
for sig in df.columns[2:]:
    signals.append(Signal(df[sig].values[1:]))
concentrations = [0, 100, 100, 100, 50, 50, 50, 25, 25, 25, 15, 15, 15, 10, 10, 10, 7.5, 7.5, 7.5, 5, 5, 5, 2.5, 2.5, 1, 1, 1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.25, 0.25, 0.25, 0.1, 0.1, 0.1, 0.1, 0.1]


Plotting some signals

In [4]:
signals[0].pplot()

# Feature extraction
 *   | Feature                   | Details | Category | Reference |
| -- | ------------------------- | ------- | --- | --------- |
| 1.  | Peak current ($I_p$)      | The peak represents the amount of electroactive species being oxidized or reduced. | Core | Laviron, J. Electroanal. Chem. (1979) & Nicholson, Anal. Chem. (1965) |
| 2.  | Peak potential ($E_p$)    | The potential corresponding for that peak current. It represents the redox potential of pyocyanin under your experimental conditions. | Core | |
| 3.  | Full width at half maximum (FWHM) | Indicates how sharp or broad the peak is. Peak broadening often increases at low concentration, noisy regimes and surface effects. $$FWHM = E_{right} - E_{left}$$ | Core | |
| 4.  | AUC | Peak area under the curve. This represents the total charge transferred during the redox event. $$AUC_{raw}=\int I(E) dE$$| Core | |
| 5. | PCA1 Component | Captures the most dominant underlying trend in the data, typically representing the primary Faradaic redox current. | Extended  | |
| 6.  | First derivative at peak | At an ideal maximum it would be zero. It will act as a quality indicator. $$\frac{dI}{dk}=\frac{I[k+1]-I[k-1]}{2\Delta E}$$ | Extended | Savitzky & Golay, Analytical Chemistry (1964) |
| 7.  | Second derivative at peak | Peak sharpness, i.e. the second derrivative at the peak. Second derivative tells you how tight the maximum is. $$\frac{d^{2}I}{dk^{2}}=\frac{I[k+1]-2I[k]+I[k-1]}{\Delta E^{2}} $$ | Extended | Savitzky & Golay, Analytical Chemistry (1964) |
| 8. | Left Slope | Indicates the rate of the leading edge, whcih correlates to the activation kinetics of the electron transfer. | Experimental | |
| 9. | Right Slope | Represents the rate of the trailing edge, which reflects the diffusion or depletion rate of the electroactive species. | Experimental | |
| 10. | Peak Asymmetry | Evaluates symmetry by considering the ratio of the left edge slope to the right edge slope. $$\frac{\|m_{left}\|}{\|m_{right}\|}$$ | Experimental | |
| 11. | Peak Sharpness | Represents the normalized intensity of the peak, indicating the electron transfer speed. $$\frac{I_p}{FWHM}$$ | Experimental | |
| 12. | Peak Compactness | Shape conformity compared to an ideal bounding box. $$\frac{AUC}{I_p \times \|E_{end} - E_{start}\|}$$ | Experimental | |
| 13. | Current Variance | Measures the dispersion of the current values across the voltammogram, reflecting the magnitude of both Faradaic and non-Faradaic processes. $$\frac{1}{N}\sum_{k=1}^{N} (I_k - \mu_I)^2$$ | Experimental | |
| 14. | Peak Skewness | Measures the horizontal asymmetry of the peak. Non-zero values indicate tailing, which can suggest non-ideal behavior or coupled chemical reactions. $$\frac{\sum I_k (E_k - \mu_E)^3}{\sum I_k \cdot \sigma_E^3}$$ | Experimental | |
| 15. | Peak Kurtosis | Measures the "peakedness" of the signal compared to a normal distribution. Positive excess kurtosis indicates sharper peaks. $$\frac{\sum I_k (E_k - \mu_E)^4}{\sum I_k \cdot \sigma_E^4} - 3$$ | Experimental | |
| 16. | Tchebichef Curve Moments | Acts as orthogonal shape descriptors. Higher-order moments quantify geometric features (such as concavity and asymmetry) without overlapping information. | Experimental | |
| 17. | Mean Peak | Provides a baseline-averaged measure of the current within the peak boundaries, acting as an indicator of total activity. $$\frac{1}{N}\sum_{E_{start}}^{E_{end}} I_k$$ | Experimental | |
| 18. | Signal Entropy | Measures the complexity of the time-domain signal. High entropy suggests noisy or erratic current behavior, while low entropy indicates structured redox peaks. $$-\sum p_i \log_2(p_i)$$ | Experimental | |
| 19. | Spectral Entropy | Evaluates the frequency distribution of the signal. Diferentiates between structured signal dominated by specific frequencies and flat, noise-like signal. $$-\sum P_f \log_2(P_f)$$ | Experimental | |
| 20. | FFT Total Power | Represents the total energy of the dynamic variations (AC components). It highlights the overall intensity of faradaic reactions. $$\frac{1}{N}\sum_{f>0} \|X(f)\|^2$$ | Experimental | |
| 21. | PCA2 Component | Isolates secondary variance patterns, which often correlate with the capacitive background charging current. | Experimental | |
| 22. | PCA3 Component | Captures tertiary structured variance, which can represent minor features or structured noise. | Experimental | |
| 23. | Wavelet Energy | Captures Time-frequency energy. It is highly sensitive to transient electrochemical events and sharp morphological changes that standard Fourier transforms might smooth over. $$\sum \|C(a,b)\|^2$$ | Experimental | |




In [5]:
def vectorize(idx: int, sig: Signal, job: str='core') -> list:
    vector = []
    vector.append(idx)
    if job in ['core', 'extended', 'experimental']:
        vector.append(sig.get_peak_current_value())
        vector.append(sig.get_peak_potential_value())
        vector.append(sig.get_peak_auc())
        vector.append(sig.get_peak_fwhm())
        
    if job in ['extended', 'experimental']:
        vector.append(sig.get_pca1_comp())
        vector.append(sig.get_first_derivative_max())
        vector.append(sig.get_second_derivative_min())
        
    if job == 'experimental':
        vector.append(sig.get_left_slope())
        vector.append(sig.get_right_slope())
        vector.append(sig.get_asymetry())
        vector.append(sig.get_peak_sharpness())
        vector.append(sig.get_peak_compactness())
        vector.append(sig.get_current_variance())
        vector.append(sig.get_peak_skewness())
        vector.append(sig.get_peak_kurtosis())
        vector.append(sig.get_tchebichef_curve_moments())
        vector.append(sig.get_mean_peak())
        vector.append(sig.get_signal_entropy())
        vector.append(sig.get_spectral_entropy())
        vector.append(sig.get_fft_power())
        vector.append(sig.get_pca2_comp())
        vector.append(sig.get_pca3_comp())
        vector.append(sig.get_wavelet_energy())

    return vector


csvheaders = [
    'sig_id',
    'peak_current',
    'peak_potential',
    'peak_AUC',
    'peak_FWHM',
]

csvheaders_extended = csvheaders + [
    'pca1_comp', 
    'first_derivative_max', 
    'second_derivative_min'
]
csvheaders_experimental = csvheaders_extended + [
    'left_slope',
    'right_slope',
    'asymetry',
    'peak_sharpness',
    'peak_compactness',
    'current_variance',
    'peak_skewness',
    'peak_kurtosis',
    'tchebichef_curve_moments',
    'mean_peak',
    'signal_entropy',
    'spectral_entropy',
    'fft_power',
    'pca2_comp',
    'pca3_comp',
    'wavelet_energy'
]

# adding the target variable to the headers as the last column
csvheaders.append('concentration')
csvheaders_extended.append('concentration')
csvheaders_experimental.append('concentration')

directory = "vectorized"
if not os.path.exists(directory):
    os.makedirs(directory)

csvfile = open('vectorized/core.csv', mode='w', newline='')
csv_writter = csv.writer(csvfile)

csv_writter.writerow(csvheaders)
for i, sig in enumerate(signals):
    signal_vector = vectorize(i, sig, job='core')
    signal_vector.append(concentrations[i + 1])
    csv_writter.writerow(signal_vector)
    print(f"signal {i}: done")

csvfile.close()

# Extended features
csvfile = open('vectorized/extended.csv', mode='w', newline='')
csv_writter = csv.writer(csvfile)

csv_writter.writerow(csvheaders_extended)
for i, sig in enumerate(signals):
    signal_vector = vectorize(i, sig, job='extended')
    signal_vector.append(concentrations[i + 1])
    csv_writter.writerow(signal_vector)
    print(f"signal {i}: done")

csvfile.close()

# Experimental features
csvfile = open('vectorized/experimental.csv', mode='w', newline='')
csv_writter = csv.writer(csvfile)

csv_writter.writerow(csvheaders_experimental)
for i, sig in enumerate(signals):
    signal_vector = vectorize(i, sig, job='experimental')
    signal_vector.append(concentrations[i + 1])
    csv_writter.writerow(signal_vector)
    print(f"signal {i}: done")

csvfile.close()

signal 0: done
signal 1: done
signal 2: done
signal 3: done
signal 4: done
signal 5: done
signal 6: done
signal 7: done
signal 8: done
signal 9: done
signal 10: done
signal 11: done
signal 12: done
signal 13: done
signal 14: done
signal 15: done
signal 16: done
signal 17: done
signal 18: done
signal 19: done
signal 20: done
signal 21: done
signal 22: done
signal 23: done
signal 24: done
signal 25: done
signal 26: done
signal 27: done
signal 28: done
signal 29: done
signal 30: done
signal 31: done
signal 32: done
signal 33: done
signal 34: done
signal 35: done
signal 36: done
signal 37: done
signal 38: done
signal 39: done
signal 0: done
signal 1: done
signal 2: done
signal 3: done
signal 4: done
signal 5: done
signal 6: done
signal 7: done
signal 8: done
signal 9: done
signal 10: done
signal 11: done
signal 12: done
signal 13: done
signal 14: done
signal 15: done
signal 16: done
signal 17: done
signal 18: done
signal 19: done
signal 20: done
signal 21: done
signal 22: done
signal 23: d